# Análise exploratória dos zeros de docentes e turmas

Objetivo: entender o padrão de `QT_DOC_FUND_AI = 0` e `QT_TUR_FUND_AI = 0` antes de decidir qualquer tratamento.


## 1. Gerar relatório e carregar os resultados

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from investigar_zeros_doc_turmas import investigate

report = investigate()
report


## 2. Comparação entre os grupos

In [ ]:
import pandas as pd

group_profile = pd.read_csv(
    PROJECT_ROOT / "results" / "zeros_doc_turmas_por_grupo.csv",
    encoding="utf-8-sig",
)
group_profile


## 3. Taxa por localização

In [ ]:
data = pd.read_csv(
    PROJECT_ROOT / "data" / "processed" / "censo_escolar_2024_municipal_ativa_anos_iniciais.csv",
    sep=";",
    encoding="utf-8-sig",
)
data["zero_doc_turma"] = data["QT_DOC_FUND_AI"].eq(0) & data["QT_TUR_FUND_AI"].eq(0)
data["localizacao"] = data["TP_LOCALIZACAO"].map({1: "Urbana", 2: "Rural"})

location_rates = (
    data.groupby("localizacao")["zero_doc_turma"]
    .agg(escolas="size", taxa_zeros="mean")
    .reset_index()
)
location_rates["taxa_zeros"] = location_rates["taxa_zeros"] * 100
location_rates


In [ ]:
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(location_rates["localizacao"], location_rates["taxa_zeros"], color=["#2F6690", "#D1495B"])
ax.set_title("Taxa de escolas com zero docentes e turmas por localização")
ax.set_ylabel("Escolas com zero (%)")
ax.set_xlabel("Localização")
for i, value in enumerate(location_rates["taxa_zeros"]):
    ax.text(i, value + 1, f"{value:.1f}%", ha="center")
plt.tight_layout()
plt.show()


## 4. Distribuição de matrículas nos dois grupos

In [ ]:
data["grupo"] = data["zero_doc_turma"].map({True: "Zeros em docentes/turmas", False: "Algum docente/turma"})
fig, ax = plt.subplots(figsize=(8, 4))
data.boxplot(column="QT_MAT_FUND_AI", by="grupo", ax=ax, showfliers=False)
ax.set_title("Distribuição de matrículas dos anos iniciais")
ax.set_xlabel("")
ax.set_ylabel("Matrículas")
fig.suptitle("")
plt.tight_layout()
plt.show()


## 5. Taxa por unidade federativa

In [ ]:
by_uf = pd.read_csv(
    PROJECT_ROOT / "results" / "zeros_doc_turmas_por_uf.csv",
    encoding="utf-8-sig",
)
top_uf = by_uf.sort_values("percentual_zeros", ascending=False).head(10)
top_uf[["SG_UF", "escolas", "zeros_doc_turma", "percentual_zeros"]]


## 6. Decisão provisória

Os zeros são mantidos como observados. Eles não serão transformados automaticamente em nulos nem imputados. Como o padrão é fortemente rural e aparece em escolas com matrículas, docentes e turmas positivas no Ensino Fundamental geral, as variáveis específicas de docentes e turmas dos anos iniciais ficam fora da primeira modelagem até uma validação conceitual adicional.
